# 04. Almacenamiento óptimo

La consulta será: Un operador portuario quiere conocer los buques que pasaron ayer por su zona. Para esto, vamos a considerar un rango de coordenada,s y estas se utilizarán para agrupar los datos.

In [0]:
#Librerias a usar
from pyspark.sql import functions as f
from pyspark.sql.window import Window

In [0]:
#Rutas 
CSV_DIR = "/Volumes/workspace/default/oceanwatch/raw/csv"
RUTA_Testing = "/Volumes/workspace/default/oceanwatch/testing/"

#Funciones de comparación de archivos
def peso(ruta):
    """Bytes totales de una ruta, recursivo."""
    total = 0
    for p in dbutils.fs.ls(ruta):
        total += peso(p.path) if p.isDir() else p.size
    return total

def humano(n):
    for unidad in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:,.1f} {unidad}"
        n /= 1024
    return f"{n:,.1f} TB"

# Carga de datos, junto a su esquema

AIS_SCHEMA = """
    MMSI long,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG double,
    COG double,
    Heading double,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType int,
    Status int,
    Length int,
    Width int,
    Draft double,
    Cargo int,
    TransceiverClass string
"""
#Se leen los datos en un dataframe para CSV con el esquema declarado.
ais_csv = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(AIS_SCHEMA)
    .csv(CSV_DIR)
    .withColumn("archivo", f.element_at(f.split(f.col("_metadata.file_path"), "/"), -1))
    .withColumn("fecha", f.to_date("BaseDateTime")) #Como existen análisis que requerirán las fecha, desde la lectura se crea una columna que convierta el BaseDatetime en una fecha.
)

ais.printSchema()


root
 |-- MMSI: long (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: double (nullable = true)
 |-- COG: double (nullable = true)
 |-- Heading: double (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Length: integer (nullable = true)
 |-- Width: integer (nullable = true)
 |-- Draft: double (nullable = true)
 |-- Cargo: integer (nullable = true)
 |-- TransceiverClass: string (nullable = true)
 |-- archivo: string (nullable = true)
 |-- fecha: date (nullable = true)



Para este ejercicio se comparará el comportamiento de archivos CSV, parquet default, parquet con particiones por fechas, y delta tables sin y con cluster by y optimizacions

In [0]:
#eliminamos la carpeta en caso de ser necesario
dbutils.fs.rm("/Volumes/workspace/default/oceanwatch/testing/", True)

#-----------Parquet sin layout ----------------------------
#Se escribe un archivo parquet con todos los CSV.

ais_csv.write.mode("overwrite").format("parquet").save(f"{RUTA_Testing}parquet_no_layout")

ais_pq = spark.read.parquet(f"{RUTA_Testing}parquet_no_layout")

#-----------Parquet particionado por fecha ---------------
#Se escribe un archivo parquet utilizando el parquet anterior pero particionandolo por fecha
(
    ais_pq.write.mode("overwrite")
    .format("parquet")
    .partitionBy("fecha")
    .save(f"{RUTA_Testing}parquet_fecha_layout")
)

#----------Delta sin layout (tabla en Unity Catalog) -----------
#Se escribe una tabla delta utilizando el parquet sin playout
(
    ais_pq.write.mode("overwrite")
    .format("delta")
    .save(f"{RUTA_Testing}delta_tabla")
)


In [0]:
#----------Estudio en peso de los archivos------------

csv_bytes = peso(CSV_DIR)
parquet_no_layouy_bytes = peso(f"{RUTA_Testing}parquet_no_layout")
parquet_fecha_layout_bytes = peso(f"{RUTA_Testing}parquet_fecha_layout")
delta_bytes = peso(f"{RUTA_Testing}delta_tabla")    

print("--------------------------------------------------------")
print(f"CSV en disco: {humano(csv_bytes)}")
print(f"Parquet sin layout: {humano(parquet_no_layouy_bytes)}")
print(f"Parquet particionado por fecha: {humano(parquet_fecha_layout_bytes)}")
print(f"Delta sin layout: {humano(delta_bytes)}")

--------------------------------------------------------
CSV en disco: 6.0 GB
Parquet sin layout: 1.7 GB
Parquet particionado por fecha: 1.7 GB
Delta sin layout: 1.8 GB


In [0]:
#------------Estudio en número de archivo de los archivos----------------

def contar_archivos(ruta, extension):
    """Cuenta los archivos con esa extensión bajo una ruta (recursivo)."""
    n = 0
    for item in dbutils.fs.ls(ruta):
        if item.isDir():
            n += contar_archivos(item.path, extension)  
        elif item.name.endswith(extension):
            n += 1
    return n


csv_archivos = sum(
    1 for i in dbutils.fs.ls(CSV_DIR)
    if i.name.startswith("AIS_") and i.name.endswith(".csv")
)
parquet_no_layout_archivos    = contar_archivos(f"{RUTA_Testing}parquet_no_layout", ".parquet")
parquet_fecha_layout_archivos = contar_archivos(f"{RUTA_Testing}parquet_fecha_layout", ".parquet")
delta_archivos = contar_archivos(f"{RUTA_Testing}delta_tabla", ".parquet")

print(f"CSV:                {csv_archivos} archivos")
print(f"Parquet sin layout: {parquet_no_layout_archivos} archivos")
print(f"Parquet por fecha:  {parquet_fecha_layout_archivos} archivos")
print(f"Delta sin layout:   {delta_archivos} archivos")

CSV:                7 archivos
Parquet sin layout: 49 archivos
Parquet por fecha:  47 archivos
Delta sin layout:   16 archivos


De acuerdo con lo evaluado, se escoge utilizar una tabla delta en vez de almacenamiento parquet

In [0]:
import time

DIA = "2023-06-03" #Día a consultar

ZONA = {"lat_min": 38.90, "lat_max": 39.05, "lon_min": -76.55, "lon_max": -76.35}   #Definición de un cuadrado de coordenadas para evaluar la zona

def consulta_operador(df):
    """Q1: posiciones de un día dentro de la zona."""
    return df.filter(
        (f.col("fecha") == f.lit(DIA).cast("date")) #Tratamiento de formato de fecha, esta es la condición 1 que filtra por fecha dato = fechha a consultar
        & f.col("LAT").between(ZONA["lat_min"], ZONA["lat_max"]) # Condición 2 que filtra por latitud
        & f.col("LON").between(ZONA["lon_min"], ZONA["lon_max"]) # Condición 3 que filtra por longitud
    )

def medir(df, nombre):
    t0 = time.time()
    r = consulta_operador(df).agg( #Aplicación de la función de consulta sobre el dataframe, y luego agg resume filtas en una sola fila con 3 valores
        f.count("*").alias("filas"), #Conteo de número de filas
        f.countDistinct("MMSI").alias("buques"), #Conteo de número de buques 
        f.countDistinct(f.col("_metadata.file_path")).alias("archivos_con_filas"), #Conteo de archivos con filas afectadas
    ).first() #Ejecución de la consulta
    seg = round(time.time() - t0, 1) #Tiempo de medición
    print(f"{nombre}: {r['filas']:,} filas · {r['buques']} buques · "
          f"{r['archivos_con_filas']} archivos con filas · {seg} s")
    return r

In [0]:
TABLA = f"delta.`{f"{RUTA_Testing}delta_tabla"}`"

# guardar la versión actual para poder consultarla más adelante
version_antes = spark.sql(f"DESCRIBE HISTORY {TABLA}").agg(f.max("version")).first()[0]

d = spark.sql(f"DESCRIBE DETAIL {TABLA}").first() #Información de la tabla
print(f"ANTES: {d['numFiles']} archivos, {humano(d['sizeInBytes'])}")

antes = medir(spark.table(TABLA), "Delta sin layout")

ANTES: 16 archivos, 1.8 GB
Delta sin layout: 70,417 filas · 405 buques · 4 archivos con filas · 1.6 s


In [0]:
spark.sql(f"ALTER TABLE {TABLA} CLUSTER BY (fecha, LAT, LON)")   # Alter tble modifica la tabla que existe, y cluster by lo hacemos sobre latyitud y longitud debido a la agrupación que se espera para la consulta.

t0 = time.time()
try:
    display(spark.sql(f"OPTIMIZE {TABLA} FULL"))                 # reescribe los archivos agrupados. FULL reorganiza todos los datos, incluso existentes antes del ALTER
except Exception as e:
    print(f"FULL no disponible, uso OPTIMIZE normal: {e}")
    display(spark.sql(f"OPTIMIZE {TABLA}"))
print(f"OPTIMIZE tardó {time.time() - t0:,.0f} s") #tiuempo que tardó la optimización

path,metrics
dbfs:/Volumes/workspace/default/oceanwatch/testing/delta_tabla,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 16, 0, false, 0, 0, 1790319513470, 1790319519145, 8, 0, null, List(0, 0), null, 19, 19, 0, 0, List(1911908734, true, false, true, null, null, null, null, 0, 7, 852792046, 852792046, 9, 1059116688, 1059116688, null, log, 16777216, 268435456, 8, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(117, 16, 0, 0, 0, 3535), 2, 1, 5, sizeAware, hierarchicalSplitStrategy, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"
dbfs:/Volumes/workspace/default/oceanwatch/testing/delta_tabla,"List(1, 2, List(235762330, 235762330, 2.3576233E8, 1, 235762330), List(117187765, 118545563, 1.17866664E8, 2, 235733328), 0, null, null, 0, 1, 16, 2, true, 0, 0, 1790319519215, 1790319527470, 8, 1, null, List(0, 0), null, 19, 19, 4623, 0, List(1911908734, false, false, false, 0.0, List(0.0, 0.0, 0.0), 0.1233106735438167, post-optimize-compaction, 0, 6, 718026690, 718026690, 8, 940188826, 940188826, null, null, 134217728, 268435456, 0, 0, 0, 0, List(1, 0, 0), 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 235733328, 0, 235733328, 0, 0, 0, 0, 0, 0, 0, 0, 235733328, 235733328, List(0, 0, 790, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"
dbfs:/Volumes/workspace/default/oceanwatch/testing/delta_tabla,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 15, 0, true, 0, 0, 1790319527541, 1790319527950, 8, 0, null, List(0, 0), SNAPPY, 19, 19, 0, 0, null, null, 0)"


OPTIMIZE tardó 16 s


In [0]:
d = spark.sql(f"DESCRIBE DETAIL {TABLA}").first()
print(f"DESPUÉS: {d['numFiles']} archivos, {humano(d['sizeInBytes'])}")
print(f"clustering: {d['clusteringColumns']}")

despues = medir(spark.table(TABLA), "Delta CLUSTER BY + OPTIMIZE")


DESPUÉS: 15 archivos, 1.8 GB
clustering: ['fecha', 'LAT', 'LON']
Delta CLUSTER BY + OPTIMIZE: 70,417 filas · 405 buques · 4 archivos con filas · 0.9 s
